# RITA 真假判别推理 (Kaggle)

## 跑之前
1. **Settings → Accelerator** 选 `GPU T4 x2` 或 `P100`
2. **Settings → Internet** 打开 (需要 clone GitHub)
3. **Add Input** 挂上两个 Dataset：
   - **数据子集**：本地用 `make_subset.py` 抽好的小子集（几十 MB），打成 zip 传上来
   - **权重包**：里面有 `best.pth`

> 本实验只需抽样子集即可，不必上传全量数据。
> 本地抽样命令见 `make_subset.py`，例如每个生成器每个 split 抽 20 张。

In [ ]:
import os, glob, torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0), '| count', torch.cuda.device_count())
print()
print('/kaggle/input 下的内容:')
for d in sorted(os.listdir('/kaggle/input')):
    p = os.path.join('/kaggle/input', d)
    try:
        print(' ', d, '->', sorted(os.listdir(p))[:8])
    except Exception:
        print(' ', d)

In [ ]:
%cd /kaggle/working
REPO = 'https://github.com/zyx-god/RITA.git'
if not os.path.exists('/kaggle/working/RITA'):
    !git clone $REPO RITA
%cd /kaggle/working/RITA
!git log --oneline -3
!ls
!pip install -q timm

In [ ]:
import glob, os
CKPT = sorted(glob.glob('/kaggle/input/**/*.pth', recursive=True))
ZIPS = sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True))
DIRS = [p for p in glob.glob('/kaggle/input/**/', recursive=True)
        if os.path.isdir(os.path.join(p, 'ADM'))]
print('权重文件 :', CKPT)
print('压缩包   :', ZIPS)
print('已解压数据根目录:', DIRS)
assert CKPT, '没找到 .pth 权重文件，检查 Add Input'

In [ ]:
import zipfile, os
DATA_ROOT = None
if ZIPS:
    DATA_ROOT = '/kaggle/working/subset'
    os.makedirs(DATA_ROOT, exist_ok=True)
    with zipfile.ZipFile(ZIPS[0]) as z:
        z.extractall(DATA_ROOT)
    # zip 内可能还套了一层目录，自动下钻
    for _ in range(3):
        if os.path.isdir(os.path.join(DATA_ROOT, 'ADM')):
            break
        subs = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))]
        if len(subs) == 1:
            DATA_ROOT = os.path.join(DATA_ROOT, subs[0])
        else:
            break
elif DIRS:
    DATA_ROOT = DIRS[0]
else:
    raise RuntimeError('没找到数据，检查 Add Input 里挂的数据集')

n = sum(len(fs) for _, _, fs in os.walk(DATA_ROOT))
size = sum(os.path.getsize(os.path.join(dp, f))
           for dp, _, fs in os.walk(DATA_ROOT) for f in fs) / 1024**2
print('DATA_ROOT =', DATA_ROOT)
print('生成器目录:', sorted(d for d in os.listdir(DATA_ROOT)
                           if os.path.isdir(os.path.join(DATA_ROOT, d))))
print(f'图片总数 = {n}, 总大小 = {size:.1f} MB')

In [ ]:
# 小子集直接一次跑完（几百张图，GPU 上几十秒）
!cd /kaggle/working/RITA && python -W ignore predict_realfake.py \
    --ckpt {CKPT[0]} \
    --root {DATA_ROOT} \
    --batch 16 --workers 2 \
    --out /kaggle/working/preds.csv \
    --calibrate

---
### 结果判读

看 **"每个生成器 AUC"**：
- 平均 AUC 明显高于 0.5（0.7+）→ RITA 在这些生成器域上有信号，值得做融合
- 平均 AUC 接近 0.5 → RITA 对它们是 OOD 的，融合不会带来增益，及时止损

两个注意点：
1. **每个生成器的样本量要够**。10 Real + 10 Fake 时 AUC 的标准误约 0.09，只能做定性判断；
   要写进报告建议 50~100 每次 split。
2. **Real 图在原始数据集里是所有生成器共享的同一批**，所以整体 ACC 会虚高，
   **以各生成器 AUC 为准**，不要报整体 ACC。

In [ ]:
import pandas as pd
df = pd.read_csv('/kaggle/working/preds.csv')
print('总样本:', df.shape[0])
print()
print(df.groupby(['generator', 'split'])['p_tamper_mean']
        .agg(['count', 'mean', 'median']).round(4))

print()
print('--- AR 序列健康度 ---')
print('n_steps 分布:')
print(df['n_steps'].value_counts().sort_index().to_string())
print()
print('final_eos_frac 均值: %.3f' % df['final_eos_frac'].mean())
print('final_eos_frac < 0.5 的样本: %d / %d (%.1f%%)'
      % ((df['final_eos_frac'] < 0.5).sum(), len(df),
         100 * (df['final_eos_frac'] < 0.5).mean()))
print()
print('判读: final_eos_frac 接近 1 = 序列正常终止；明显偏低 = 没收敛，')
print('      这时用 --max_steps 6 再跑一次，看 n_steps 是稳定在 3 还是一路顶到 6。')
df.head()

### 结果下载
右侧 **Output** 面板可下载 `/kaggle/working/preds.csv`。

CSV 字段：`generator, split, is_fake, category, path, p_tamper_mean, tamper_ratio, p_tamper_p95, n_steps`

- `p_tamper_mean`：连续分数，算 AUC 用它
- `tamper_ratio`：硬判出的"篡改面积比"，定阈值用它
- `n_steps`：AR 收敛用了多少步；如果普遍是 1 或普遍顶到上限，说明序列退化，结果不可信